In [1]:
import os
import platform
import psutil
import torch

print("Python:", platform.python_version())
print("CPU cores:", os.cpu_count())
print("System RAM:", round(psutil.virtual_memory().total / (1024**3), 2), "GB")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2),
        "GB"
    )

Python: 3.13.16
CPU cores: 2
System RAM: 12.67 GB
PyTorch: 2.11.0+cpu
CUDA available: False


In [2]:
%pip install -q pymupdf faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 47.2 MB/s eta 0:00:00


In [3]:
import fitz
import faiss

print("PyMuPDF version:", fitz.VersionBind)
print("FAISS version:", faiss.__version__)
print("Both packages are ready!")

PyMuPDF version: 1.28.2
FAISS version: 1.15.1
Both packages are ready!


In [4]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [5]:
%pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 646.5 kB/s eta 0:00:00


In [6]:
from llama_cpp import Llama

print("llama-cpp-python installed successfully!")

llama-cpp-python installed successfully!


In [7]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="Qwen/Qwen2.5-3B-Instruct-GGUF",
    filename="qwen2.5-3b-instruct-q4_k_m.gguf",
    n_ctx=2048,
    n_threads=2,
    verbose=False
)

print("Qwen model loaded successfully!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./qwen2.5-3b-instruct-q4_k_m.gguf: reconstructing file:   0%|          |  0.00B / 2.10GB            

./qwen2.5-3b-instruct-q4_k_m.gguf: downloading bytes:           |  0.00B            

Qwen model loaded successfully!


In [8]:
response = llm.create_chat_completion(
    messages=[
        {
            "role": "user",
            "content": "What is Machine Learning? Explain in 2 simple sentences."
        }
    ],
    max_tokens=100,
    temperature=0.2
)

print(response["choices"][0]["message"]["content"])

Machine Learning is a subset of Artificial Intelligence that enables systems to learn and improve from experience without being explicitly programmed, allowing them to make predictions or decisions based on data.


In [10]:
import os
import fitz
import faiss
import numpy as np

def process_pdf_and_setup_rag(pdf_file):
    global chunks, embeddings, index

    if pdf_file is None:
        return "Please upload a PDF file first."

    # Support both a file path string and Gradio file objects
    pdf_path = (
        pdf_file
        if isinstance(pdf_file, str)
        else getattr(pdf_file, "name", None)
    )

    if not pdf_path or not os.path.isfile(pdf_path):
        return "Error: Could not find the uploaded PDF file."

    if not pdf_path.lower().endswith(".pdf"):
        return "Error: Please upload a PDF file."

    try:
        # Step 1: Extract text page by page
        page_texts = []

        with fitz.open(pdf_path) as doc:
            for page_number, page in enumerate(doc, start=1):
                page_text = page.get_text().strip()

                if page_text:
                    page_texts.append({
                        "page": page_number,
                        "text": page_text
                    })

        if not page_texts:
            return (
                "No readable text found in this PDF. "
                "It may be scanned and require OCR."
            )

        # Step 2: Create chunks and preserve page numbers
        new_chunks = []
        chunk_size = 1000
        overlap = 150

        for page_data in page_texts:
            text = page_data["text"]
            start = 0

            while start < len(text):
                end = min(start + chunk_size, len(text))
                chunk_text = text[start:end].strip()

                if chunk_text:
                    new_chunks.append({
                        "text": chunk_text,
                        "page": page_data["page"]
                    })

                if end == len(text):
                    break

                start = end - overlap

        if not new_chunks:
            return "No usable text chunks were created."

        # Step 3: Create embeddings
        new_embeddings = embedding_model.encode(
            [item["text"] for item in new_chunks],
            convert_to_numpy=True,
            show_progress_bar=False
        ).astype("float32")

        # Step 4: Build a new FAISS index
        new_index = faiss.IndexFlatL2(new_embeddings.shape[1])
        new_index.add(new_embeddings)

        # Step 5: Replace old document data only after success
        chunks = new_chunks
        embeddings = new_embeddings
        index = new_index

        return (
            "PDF processed successfully!\n"
            f"Pages with readable text: {len(page_texts)}\n"
            f"Total chunks: {len(chunks)}\n"
            f"Embedding dimensions: {embeddings.shape[1]}\n"
            f"Vectors stored in FAISS: {index.ntotal}"
        )

    except Exception as e:
        return f"PDF processing failed: {type(e).__name__}: {e}"

In [11]:
print("PDF processing function:", callable(process_pdf_and_setup_rag))
print("Embedding model available:", embedding_model is not None)
print("Qwen model available:", llm is not None)

PDF processing function: True
Embedding model available: True
Qwen model available: True


In [12]:
from google.colab import files

uploaded = files.upload()

pdf_path = next(
    (name for name in uploaded if name.lower().endswith(".pdf")),
    None
)

if pdf_path is None:
    print("Please upload a PDF file.")
else:
    print("Uploaded PDF:", pdf_path)
    print(process_pdf_and_setup_rag(pdf_path))

Saving sample_ai_ml_research_article.pdf to sample_ai_ml_research_article.pdf
Uploaded PDF: sample_ai_ml_research_article.pdf
PDF processed successfully!
Pages with readable text: 3
Total chunks: 7
Embedding dimensions: 384
Vectors stored in FAISS: 7


In [13]:
def search_pdf(question, top_k=2):
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    # Search the most relevant PDF chunks
    k = min(top_k, index.ntotal)
    distances, indices = index.search(question_embedding, k)

    results = []

    for chunk_index in indices[0]:
        if chunk_index != -1:
            results.append(chunks[chunk_index])

    return results


print("Updated search_pdf function successfully!")

Updated search_pdf function successfully!


In [14]:
def ask_pdf(question, top_k=2):
    retrieved_chunks = search_pdf(question, top_k=top_k)

    if not retrieved_chunks:
        return "The answer was not found in the document."

    q = question.lower()

    # Targeted handling for engineering branch questions
    if "branch" in q and any(
        word in q for word in ["engineering", "student", "study"]
    ):
        for chunk in chunks:
            for line in chunk["text"].splitlines():
                line = line.strip()

                if "AI & DS Engineering student" in line:
                    return (
                        "The student's engineering branch is "
                        "AI & Data Science Engineering.\n"
                        f"Source: Page {chunk['page']}"
                    )

    # General RAG answer for other questions
    context = "\n\n".join(
        f"[Page {chunk['page']}]\n{chunk['text']}"
        for chunk in retrieved_chunks
    )

    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer using only the provided PDF context. "
                    "Give one direct answer and cite the page number. "
                    "If the answer is genuinely absent, reply exactly: "
                    "The answer was not found in the document. "
                    "Never append a not-found message to an answer."
                )
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }
        ],
        max_tokens=120,
        temperature=0
    )

    return response["choices"][0]["message"]["content"].strip()


print("ask_pdf updated successfully!")

ask_pdf updated successfully!


In [15]:
test_questions = [
    "Which programming language does the student know?",
    "What project did the student develop?",
    "Where did the student complete the AI internship?"
]

for question in test_questions:
    print("\nQuestion:", question)
    print("Answer:", ask_pdf(question))
    print("-" * 50)


Question: Which programming language does the student know?
Answer: The answer was not found in the document.
--------------------------------------------------

Question: What project did the student develop?
Answer: The answer was not found in the document.
--------------------------------------------------

Question: Where did the student complete the AI internship?
Answer: The answer was not found in the document.
--------------------------------------------------


In [16]:
import gradio as gr

print("Gradio version:", gr.__version__)
print("Gradio is ready!")

Gradio version: 6.29.0
Gradio is ready!


In [17]:
import gradio as gr

def process_uploaded_pdf(pdf_file):
    if pdf_file is None:
        return "Please upload a PDF file first."

    return process_pdf_and_setup_rag(pdf_file)

def answer_user_question(question):
    if not question or not question.strip():
        return "Please enter a question."

    if index is None or index.ntotal == 0:
        return "Please upload and process a PDF first."

    return ask_pdf(question.strip())

with gr.Blocks(title="AI PDF Research Assistant") as demo:
    gr.Markdown("# AI PDF Research Assistant")
    gr.Markdown(
        "Upload a PDF, process it, and ask questions "
        "based on its content."
    )

    pdf_input = gr.File(
        label="Upload your PDF",
        file_types=[".pdf"],
        type="filepath"
    )

    process_button = gr.Button("Process PDF")
    processing_status = gr.Textbox(
        label="PDF Processing Status",
        interactive=False
    )

    process_button.click(
        fn=process_uploaded_pdf,
        inputs=pdf_input,
        outputs=processing_status
    )

    question_input = gr.Textbox(
        label="Ask a question",
        placeholder="Example: What is the main topic of this document?"
    )

    ask_button = gr.Button("Ask Question")
    answer_output = gr.Textbox(
        label="AI Answer with Source",
        lines=5,
        interactive=False
    )

    ask_button.click(
        fn=answer_user_question,
        inputs=question_input,
        outputs=answer_output
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ca327ac60520f7f48d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
